In [1]:
# 📦 Step 1: Import libraries
import numpy as np
import pandas as pd
import pickle
import os

In [2]:
# 📂 Step 2: Load training data
file_path = "./data/train.csv"  # Update if path differs

df = pd.read_csv(file_path)
df.head()

,POSTED_BY,UNDER_CONSTRUCTION,RERA,BHK_NO.,BHK_OR_RK,SQUARE_FT,READY_TO_MOVE,RESALE,ADDRESS,LONGITUDE,LATITUDE,TARGET(PRICE_IN_LACS)
0,Owner,0,0,2,BHK,1300.236407,1,1,"Ksfc Layout,Bangalore",12.969910,77.597960,55.0
1,Dealer,0,0,2,BHK,1275.000000,1,1,"Vishweshwara Nagar,Mysore",12.274538,76.644605,51.0
2,Owner,0,0,2,BHK,933.159722,1,1,"Jigani,Bangalore",12.778033,77.632191,43.0
3,Owner,0,1,2,BHK,929.921143,1,1,"Sector-1 Vaishali,Ghaziabad",28.642300,77.344500,62.5
4,Dealer,1,0,2,BHK,999.009247,0,1,"New Town,Kolkata",22.592200,88.484911,60.5


In [4]:
def one_hot_encode(series, categories):
    full_names = [f"{series.name}_{cat}" for cat in categories]
    return pd.get_dummies(series, prefix=series.name)[full_names]

# One-hot encode 'POSTED_BY' and 'BHK_OR_RK'
posted_by_encoded = one_hot_encode(df["POSTED_BY"], ["Owner", "Dealer", "Builder"])
bhk_or_rk_encoded = one_hot_encode(df["BHK_OR_RK"], ["BHK", "RK"])

# Combine all relevant features
features = pd.concat([
    posted_by_encoded,
    df[["UNDER_CONSTRUCTION", "RERA", "BHK_NO.", "SQUARE_FT", "READY_TO_MOVE", "RESALE"]],
    bhk_or_rk_encoded,
    df[["LONGITUDE", "LATITUDE"]]
], axis=1)

X = features.values
y = df["TARGET(PRICE_IN_LACS)"].values


In [7]:
X = features.values.astype(np.float64)  # force correct dtype

X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X_norm = (X - X_mean) / X_std
X_b = np.c_[np.ones((X_norm.shape[0], 1)), X_norm]

In [8]:
# 🤖 Step 5: Train Linear Regression using Gradient Descent
theta = np.zeros(X_b.shape[1])
lr = 0.01
epochs = 1000

for epoch in range(epochs):
    predictions = X_b.dot(theta)
    error = predictions - y
    gradients = 2 / len(X_b) * X_b.T.dot(error)
    theta -= lr * gradients

print("✅ Training complete. Final theta:", theta)


✅ Training complete. Final theta: [ 142.89874547   -6.76813064   27.3333016   -69.65755747   -7.56035711
    4.6532383    71.82348085  264.18371341    7.56035711 -175.3724557
    0.97776824   -0.97776824  -25.19257353  -15.10598734]


In [9]:
# 💾 Step 6: Save model to models/linear_model.pkl
model = {
    "theta": theta,
    "mean": X_mean,
    "std": X_std
}

os.makedirs("models", exist_ok=True)
with open("models/linear_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("✅ Model saved to models/linear_model.pkl")


✅ Model saved to models/linear_model.pkl


In [17]:
# 🔍 Predict price for a new input row
def predict(sample_dict):
    # Manual one-hot
    posted_by = [int(sample_dict["POSTED_BY"] == c) for c in ["Owner", "Dealer", "Builder"]]
    bhk_or_rk = [int(sample_dict["BHK_OR_RK"] == c) for c in ["BHK", "RK"]]

    input_features = posted_by + [
        int(sample_dict["UNDER_CONSTRUCTION"]),
        int(sample_dict["RERA"]),
        int(sample_dict["BHK_NO."]),
        float(sample_dict["SQUARE_FT"]),
        int(sample_dict["READY_TO_MOVE"]),
        int(sample_dict["RESALE"])
    ] + bhk_or_rk + [
        float(sample_dict["LONGITUDE"]),
        float(sample_dict["LATITUDE"])
    ]

    x = np.array(input_features)
    x_norm = (x - X_mean) / X_std
    x_b = np.insert(x_norm, 0, 1)
    return x_b.dot(theta)

# Example input
sample = {
    "POSTED_BY": "Owner",
    "UNDER_CONSTRUCTION": "0",
    "RERA": "0",
    "BHK_NO.": "2",
    "BHK_OR_RK": "BHK",
    "SQUARE_FT": "1300.236407",
    "READY_TO_MOVE": "1",
    "RESALE": "1",
    "LONGITUDE": "12.96991",
    "LATITUDE": "77.59796"
}

print("Predicted price (Lacs):", predict(sample))


Predicted price (Lacs): 62.86993404236981
